# Flood Mapping Data Pipeline (Colab)

GEOG761 Group 05 — Flood Mapping & Response

Fetches the six input layers from the project's architecture diagram and
saves them under `/content/data/raw/...`, ready for the ML model step:

| Layer | Source | Format |
|---|---|---|
| Land Type | ESA WorldCover v200 (Earth Engine) | GeoTIFF |
| Elevation | Copernicus GLO-30 DEM (Earth Engine) | GeoTIFF |
| SAR Satellite | Sentinel-1 GRD (Earth Engine) | GeoTIFF (VV/VH composite) |
| Population Density | WorldPop (Earth Engine) | GeoTIFF |
| Critical Infrastructure | OpenStreetMap (LINZ swap-in ready) | GeoJSON per category |
| Transport Infrastructure | OpenStreetMap (LINZ swap-in ready) | GeoJSON per category |

Run the cells top to bottom. The Earth Engine cell below opens a one-time
browser login the first time you run this notebook in a session.

**Optional: save outputs to Google Drive** so they survive when the Colab
runtime resets — see the "Mount Google Drive (optional)" cell.


## 1. Setup

In [ ]:
# Runtime packages. earthengine-api and requests ship with Colab; the rest don't.
!pip install -q geemap osmnx geopandas python-dotenv


In [ ]:
import ee

# One-time interactive login per Colab session/account. If this cell has
# already been authorised for this account, it returns immediately.
ee.Authenticate()

# Use your group's GEE-registered Cloud project.
EE_PROJECT = "ayao859-geog761"
ee.Initialize(project=EE_PROJECT)
print("Earth Engine ready, project:", EE_PROJECT)


## 2. Mount Google Drive (optional)

Skip this cell if you're fine with outputs disappearing when the Colab
runtime resets. Otherwise mount Drive and outputs will be written under
`/content/drive/MyDrive/save-kiwis-from-flooding/data/raw/...` instead of
`/content/data/raw/...`.


In [ ]:
USE_DRIVE = False  # flip to True to persist outputs to Google Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")


## 3. Configuration

In [ ]:
from datetime import date, timedelta
from pathlib import Path
import os

# --- Output paths ------------------------------------------------------------
BASE_DIR = Path("/content/drive/MyDrive/save-kiwis-from-flooding") if USE_DRIVE else Path("/content")
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"

RASTER_DIRS = {
    "land_type": RAW_DIR / "land_type",
    "elevation": RAW_DIR / "elevation",
    "sar": RAW_DIR / "sar",
    "population_density": RAW_DIR / "population_density",
}
VECTOR_DIRS = {
    "critical_infrastructure": RAW_DIR / "critical_infrastructure",
    "transport_infrastructure": RAW_DIR / "transport_infrastructure",
}
for _dir in (*RASTER_DIRS.values(), *VECTOR_DIRS.values()):
    _dir.mkdir(parents=True, exist_ok=True)

# --- Area of interest: whole of New Zealand -----------------------------------
COUNTRY_NAME = "New Zealand"
GEE_COUNTRY_TABLE = "USDOS/LSIB_SIMPLE/2017"
GEE_COUNTRY_NAME_FIELD = "country_na"

countries = ee.FeatureCollection(GEE_COUNTRY_TABLE)
NZ_BOUNDARY = countries.filter(ee.Filter.eq(GEE_COUNTRY_NAME_FIELD, COUNTRY_NAME)).geometry()

# NZ's 16 official regions -- used to chunk OSM/Overpass queries so a single
# request never spans the whole country (Overpass rejects/times out on
# country-scale `highway`/`building` queries for NZ).
NZ_REGIONS = [
    "Northland Region, New Zealand", "Auckland Region, New Zealand",
    "Waikato Region, New Zealand", "Bay of Plenty Region, New Zealand",
    "Gisborne Region, New Zealand", "Hawke's Bay Region, New Zealand",
    "Taranaki Region, New Zealand", "Manawatu-Whanganui Region, New Zealand",
    "Wellington Region, New Zealand", "Tasman Region, New Zealand",
    "Nelson Region, New Zealand", "Marlborough Region, New Zealand",
    "West Coast Region, New Zealand", "Canterbury Region, New Zealand",
    "Otago Region, New Zealand", "Southland Region, New Zealand",
]

# --- Event window --------------------------------------------------------------
# Sentinel-1 revisit over NZ is ~12 days; default to a rolling 24-day window.
# Set these explicitly to target a specific cyclone/flood event instead.
END_DATE = date.today()
START_DATE = END_DATE - timedelta(days=24)
# START_DATE, END_DATE = date(2026, 2, 10), date(2026, 2, 20)  # example override

# --- LINZ Data Service (optional) ----------------------------------------------
# Set LINZ_API_KEY to fetch Critical/Transport Infrastructure from LINZ
# instead of the OpenStreetMap fallback. Free key: sign in at
# https://data.linz.govt.nz -> profile menu -> "API Keys" -> "Generate API
# key", enabling the "Web Feature Service (WFS)" scope.
LINZ_API_KEY = os.environ.get("LINZ_API_KEY")  # or paste directly for a quick test
LINZ_WFS_BASE = "https://data.linz.govt.nz/services"

# --- Resolution ------------------------------------------------------------
# Country-wide exports default to a coarser scale to keep exports tractable;
# lower this for a smaller AOI / higher-resolution local test.
SCALE_M = 100

print("AOI: whole of", COUNTRY_NAME)
print("SAR window:", START_DATE, "to", END_DATE)
print("Outputs under:", RAW_DIR)


## 4. Land Type — ESA WorldCover v200

10 m resolution, 11 land-cover classes (Sentinel-1/2 derived). Matches the
architecture diagram's "Land Type" box directly — no source substitution
needed.


In [ ]:
import geemap

WORLDCOVER_COLLECTION = "ESA/WorldCover/v200"
LAND_TYPE_CLASSES = {
    10: "Tree cover", 20: "Shrubland", 30: "Grassland", 40: "Cropland",
    50: "Built-up", 60: "Bare / sparse vegetation", 70: "Snow and ice",
    80: "Permanent water bodies", 90: "Herbaceous wetland",
    95: "Mangroves", 100: "Moss and lichen",
}

def fetch_land_cover(aoi):
    image = ee.ImageCollection(WORLDCOVER_COLLECTION).first()
    return image.select("Map").clip(aoi).rename("land_type")

land_type_image = fetch_land_cover(NZ_BOUNDARY)
land_type_path = str(RASTER_DIRS["land_type"] / "nz_land_type.tif")
geemap.ee_export_image(
    land_type_image, filename=land_type_path, scale=SCALE_M, region=NZ_BOUNDARY, file_per_band=False
)
print("Saved:", land_type_path)


## 5. Elevation

Target source per the architecture diagram is the LINZ 8 m DEM (used for
the "bathtub model": ground elevation − SAR flood-extent boundary → water
surface elevation → depth). LINZ doesn't publish that DEM through Earth
Engine or a queryable API — it needs a manual export from
data.linz.govt.nz. This notebook defaults to the Copernicus GLO-30 DEM
(30 m, global, free via Earth Engine) and leaves a swap-in cell below.


In [ ]:
COPERNICUS_DEM_COLLECTION = "COPERNICUS/DEM/GLO30"

def fetch_elevation(aoi):
    dem = ee.ImageCollection(COPERNICUS_DEM_COLLECTION).select("DEM").mosaic()
    return dem.clip(aoi).rename("elevation_m")

elevation_image = fetch_elevation(NZ_BOUNDARY)
elevation_path = str(RASTER_DIRS["elevation"] / "nz_elevation.tif")
geemap.ee_export_image(
    elevation_image, filename=elevation_path, scale=SCALE_M, region=NZ_BOUNDARY, file_per_band=False
)
print("Saved:", elevation_path)


In [ ]:
# Swap-in point once a LINZ 8 m DEM GeoTIFF has been downloaded locally
# (requires a LINZ API key + manual export from data.linz.govt.nz):
#
#   import rioxarray
#   linz_dem = rioxarray.open_rasterio("/content/nz_8m_dem.tif")
#
# Use `linz_dem` instead of `elevation_image` for higher-resolution depth
# modelling once you have it.


## 6. SAR Satellite — Sentinel-1 GRD

Chosen per the group's satellite comparison (free/open access, 10 m
resolution, ~12-day revisit over NZ, used in operational systems like
Copernicus Global Flood Monitoring). Includes a baseline Otsu-style VV
threshold water mask as a sanity check against the ML model's predictions.


In [ ]:
S1_COLLECTION = "COPERNICUS/S1_GRD"

def fetch_sar_collection(aoi, start_date, end_date, orbit_pass=None):
    collection = (
        ee.ImageCollection(S1_COLLECTION)
        .filterBounds(aoi)
        .filterDate(str(start_date), str(end_date))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .filter(ee.Filter.eq("instrumentMode", "IW"))
    )
    if orbit_pass:
        collection = collection.filter(ee.Filter.eq("orbitProperties_pass", orbit_pass))
    return collection

def to_water_mask(image, vv_threshold_db=-17.0):
    """Baseline water mask -- a placeholder for the real Otsu/adaptive
    threshold the ML model will compute, kept here as a sanity check."""
    return image.select("VV").lt(vv_threshold_db).rename("water_mask")

sar_collection = fetch_sar_collection(NZ_BOUNDARY, START_DATE, END_DATE)
sar_composite = sar_collection.select(["VV", "VH"]).median().clip(NZ_BOUNDARY)

sar_path = str(RASTER_DIRS["sar"] / f"nz_sar_{START_DATE}_to_{END_DATE}.tif")
geemap.ee_export_image(
    sar_composite, filename=sar_path, scale=SCALE_M, region=NZ_BOUNDARY, file_per_band=False
)
print("Saved:", sar_path)
print("Scenes used:", sar_collection.size().getInfo())


## 7. Population Density — WorldPop

Architecture diagram target: WorldPop (NZ, ~100 m) or the Stats NZ Census
meshblock grid. WorldPop is used here since it's published directly as a
GEE ImageCollection; the Stats NZ grid would need a separate download +
join step.


In [ ]:
WORLDPOP_COLLECTION = "WorldPop/GP/100m/pop"
WORLDPOP_YEAR = 2020

def fetch_population(aoi, year=WORLDPOP_YEAR):
    collection = (
        ee.ImageCollection(WORLDPOP_COLLECTION)
        .filter(ee.Filter.eq("country", "NZL"))
        .filter(ee.Filter.eq("year", year))
    )
    return collection.mosaic().clip(aoi).rename("population_density")

population_image = fetch_population(NZ_BOUNDARY)
population_path = str(RASTER_DIRS["population_density"] / f"nz_population_{WORLDPOP_YEAR}.tif")
geemap.ee_export_image(
    population_image, filename=population_path, scale=SCALE_M, region=NZ_BOUNDARY, file_per_band=False
)
print("Saved:", population_path)


## 8. Critical Infrastructure

Architecture diagram target sources: LINZ + Stats NZ / Civil Defence NZ,
which need a LINZ API key (see the LINZ section below). Until a key is
available, each category is fetched from OpenStreetMap instead — open
access, nationwide coverage, and a reasonable tag-for-tag match.

Runs region-by-region across NZ's 16 regions (country-wide Overpass queries
for dense tags are routinely rejected or time out) — **this cell can take a
few minutes.**


In [ ]:
import time
import geopandas as gpd
import pandas as pd
import osmnx as ox

try:
    from osmnx._errors import InsufficientResponseError
except ImportError:
    class InsufficientResponseError(Exception):
        pass

CRITICAL_INFRA_TAGS = {
    "hospitals_medical": {"amenity": ["hospital", "clinic"]},
    "emergency_services": {"amenity": ["fire_station", "police"], "emergency": ["ambulance_station"]},
    "power_substations": {"power": ["substation", "plant"]},
    "water_treatment": {"man_made": ["wastewater_plant", "water_works"]},
    "telecommunications": {"man_made": ["communications_tower", "tower"]},
    "schools_shelters": {"amenity": ["school"]},
    "civil_defence_shelters": {"emergency": ["assembly_point", "shelter"]},
}

def fetch_tags_nationwide(tags, places=NZ_REGIONS, request_delay_s=1.0):
    frames = []
    for place in places:
        try:
            gdf = ox.features_from_place(place, tags)
        except InsufficientResponseError:
            gdf = None
        except Exception as exc:
            print(f"  [warn] {place}: {exc!r} -- skipping this region")
            gdf = None
        if gdf is not None and not gdf.empty:
            gdf = gdf.copy()
            gdf["region"] = place
            frames.append(gdf)
        time.sleep(request_delay_s)
    if not frames:
        return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
    return gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), crs=frames[0].crs)

def fetch_critical_infrastructure(use_linz=None):
    if use_linz is None:
        use_linz = bool(LINZ_API_KEY)
    results = {}
    for category, tags in CRITICAL_INFRA_TAGS.items():
        gdf = None
        if use_linz:
            gdf = fetch_linz_layer(category)  # defined in the LINZ section below
        if gdf is None:
            print(f"  fetching '{category}' from OpenStreetMap...")
            gdf = fetch_tags_nationwide(tags)
        gdf = gdf.copy()
        gdf["category"] = category
        results[category] = gdf
    return results

critical_infra = fetch_critical_infrastructure()
for category, gdf in critical_infra.items():
    out_path = VECTOR_DIRS["critical_infrastructure"] / f"{category}.geojson"
    gdf.to_file(out_path, driver="GeoJSON")
    print(f"  {category}: {len(gdf)} features -> {out_path}")


## 9. Transport Infrastructure

Architecture diagram target sources: LINZ, Auckland Transport /
Christchurch City open data, and Waka Kotahi NZ Transport Agency — each its
own agency with its own access process. Defaults to OpenStreetMap for the
same reason as Critical Infrastructure above.


In [ ]:
TRANSPORT_TAGS = {
    "roads_highways": {"highway": True},
    "bridges": {"bridge": True},
    "railways": {"railway": ["rail", "light_rail"]},
    "airports_airstrips": {"aeroway": ["aerodrome", "runway"]},
    "ports_harbours": {"landuse": ["harbour"]},
    "tunnels": {"tunnel": True},
    "transit_stations": {"public_transport": ["station", "stop_position"]},
}

def fetch_transport_infrastructure(use_linz=None):
    if use_linz is None:
        use_linz = bool(LINZ_API_KEY)
    results = {}
    for category, tags in TRANSPORT_TAGS.items():
        gdf = None
        if use_linz:
            gdf = fetch_linz_layer(category)  # defined in the LINZ section below
        if gdf is None:
            print(f"  fetching '{category}' from OpenStreetMap...")
            gdf = fetch_tags_nationwide(tags)
        gdf = gdf.copy()
        gdf["category"] = category
        results[category] = gdf
    return results

transport_infra = fetch_transport_infrastructure()
for category, gdf in transport_infra.items():
    out_path = VECTOR_DIRS["transport_infrastructure"] / f"{category}.geojson"
    gdf.to_file(out_path, driver="GeoJSON")
    print(f"  {category}: {len(gdf)} features -> {out_path}")


## 10. Optional: switch Critical/Transport Infrastructure to LINZ

Skip this section entirely while `LINZ_API_KEY` is unset — sections 8 and 9
already ran on OpenStreetMap above. Come back to it once you have a key:

1. Register / sign in at https://data.linz.govt.nz
2. Profile menu -> **API Keys** -> **Generate API key**, enable the **Web
   Feature Service (WFS)** scope.
3. Set `LINZ_API_KEY` in the Configuration cell above (section 3).
4. Look up each dataset's numeric `layer-XXXXX` id on its LINZ data page and
   fill it into `LINZ_LAYER_IDS` below.
5. Re-run the Critical/Transport Infrastructure cells (8 and 9) — they call
   `fetch_linz_layer` automatically once `LINZ_API_KEY` is set.


In [ ]:
import requests

# Fill in with confirmed layer ids from https://data.linz.govt.nz once a key
# is available. Left as None so `fetch_linz_layer` fails loudly instead of
# silently querying the wrong layer.
LINZ_LAYER_IDS = {
    "roads_highways": None,   # e.g. NZ Roads (Topo)
    "railways": None,         # e.g. NZ Railway Centrelines (Topo)
    "hospitals_medical": None,
    # add the rest of the categories from sections 8/9 as you confirm their layer ids
}

def fetch_linz_layer(category, bbox=None):
    """Fetch a LINZ vector layer via WFS. Returns None (not an error) so the
    calling code can fall back to OpenStreetMap for any unconfigured category."""
    if not LINZ_API_KEY:
        return None
    layer_id = LINZ_LAYER_IDS.get(category)
    if layer_id is None:
        print(f"  [info] no LINZ layer id configured for '{category}' -- using OpenStreetMap")
        return None

    url = f"{LINZ_WFS_BASE};key={LINZ_API_KEY}/wfs"
    params = {
        "service": "WFS", "version": "2.0.0", "request": "GetFeature",
        "typeNames": f"layer-{layer_id}", "outputFormat": "json", "srsName": "EPSG:4326",
    }
    if bbox:
        params["bbox"] = ",".join(str(v) for v in bbox)
    response = requests.get(url, params=params, timeout=120)
    response.raise_for_status()
    return gpd.GeoDataFrame.from_features(response.json()["features"], crs="EPSG:4326")


## 11. Summary

All six layers are now under `data/raw/` (or your mounted Drive folder if
`USE_DRIVE = True`), ready for the ML preprocessing step.


In [ ]:
for name, path in {**RASTER_DIRS, **VECTOR_DIRS}.items():
    files = sorted(p.name for p in path.glob("*"))
    print(f"{name}: {files}")
